# Multi-Modal DistilBERT Yelp Fake Detector — **FOCAL LOSS** variant of Model B (tuned)

Same multi-modal architecture as Model B's DistilBERT (DistilBERT [CLS] ⊕ behavioural MLP →
fusion → 1 logit), trained on the **pure full** Yelp data (`yelp_multimodal_features_full.csv`),
but with **binary focal loss** (Lin et al. 2017) instead of weighted BCE — the Model-B
counterpart of the text-only focal ablation. Focal down-weights easy examples (γ) and α weights
the minority fake class.

**Best-shot / tuned:** rather than hard-set γ=2 / α=0.89, this notebook **sweeps** focal
hyperparameters — **Phase A** tunes (γ, α) on a fast proxy (a 120k stratified train subsample ×
2 epochs, scored on the full balanced val set), then **Phase B** retrains from scratch at the
best config on the **full** 642k train, full epochs. Selection is by val macro-F1; the chosen
(γ, α) are recorded in the exported meta. Everything else (LLRD, val-tuned threshold, early
stopping, ONNX export) is identical to standard Model B, so it's serving-compatible and strictly
comparable.

**Expectation (honest):** the text-only focal ablation landed *below* baseline, and focal only
reweights the loss — it adds no new signal — so even tuned this may not beat the standard Model B
DistilBERT. The sweep is to give it a genuine best shot for the ablation table.

**Before running:** upload `data/yelp_multimodal_features_full.csv` (~480 MB) to Drive root;
**GPU** runtime (A100/L4; high-RAM advised — full 642k tokenized + a 120k proxy in memory).
**Output:** downloads `yelp_multimodal_distilbert_focal_onnx.zip` → extract to
`data/yelp_multimodal_distilbert_focal_onnx/` (feature order == `model_b.FEAT_COLS`).

In [ ]:
!pip install -q -U transformers

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertModel, DistilBertTokenizerFast
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score, classification_report
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from google.colab import drive
drive.mount('/content/drive')

CSV='/content/drive/MyDrive/yelp_multimodal_features_full.csv'   # PURE full 678k (no AI)
MAX_LEN=256; BATCH=64; EPOCHS=4; LR=2e-5     # BATCH 64 to cut steps on 642k; drop to 32 if OOM
DROP={'user_reviews_on_this_biz'}
META={'text','label','split','source','fake_type','generator','length_bucket'}
device='cuda' if torch.cuda.is_available() else 'cpu'; print('device:', device)

df=pd.read_csv(CSV); df['text']=df['text'].astype(str)
feat_cols=[c for c in df.columns if c not in META and c not in DROP]
print('rows', len(df), '| behavioural features', len(feat_cols))
tr=df[df.split=='train'].reset_index(drop=True)
va=df[df.split=='val'].reset_index(drop=True)
te=df[df.split=='test'].reset_index(drop=True)

sc=StandardScaler()
Btr=sc.fit_transform(tr[feat_cols].values).astype('float32')
Bva=sc.transform(va[feat_cols].values).astype('float32')
Bte=sc.transform(te[feat_cols].values).astype('float32')

# CLASS IMBALANCE (pure full data): train is the true ~11% fake (val/test balanced). No AI ×10 —
# instead per-sample weight = n_neg/n_pos for the minority (== BCE pos_weight); val-tuned thr sets
# the final operating point.
POS_WEIGHT=float((tr.label.values==0).sum())/max(int((tr.label.values==1).sum()),1)
sw_tr=np.where(tr.label.values==1, POS_WEIGHT, 1.0).astype('float32')
print('train fake rate', round(float(tr.label.mean()),3), '| POS_WEIGHT', round(POS_WEIGHT,2))

In [ ]:
tok=DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

class MMData(Dataset):
    def __init__(self, texts, beh, labels, weights=None):
        self.enc=tok(list(texts), truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')
        self.beh=torch.tensor(beh); self.y=torch.tensor(np.asarray(labels), dtype=torch.float32)
        self.w=torch.tensor(weights if weights is not None else np.ones(len(labels), dtype='float32'))
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return (self.enc['input_ids'][i], self.enc['attention_mask'][i],
                self.beh[i], self.y[i], self.w[i])

tr_dl=DataLoader(MMData(tr.text.values, Btr, tr.label.values, sw_tr), batch_size=BATCH, shuffle=True)
va_dl=DataLoader(MMData(va.text.values, Bva, va.label.values), batch_size=64)
te_dl=DataLoader(MMData(te.text.values, Bte, te.label.values), batch_size=64)

In [ ]:
import torch.nn.functional as F

class MultiModalDistilBert(nn.Module):
    def __init__(self, n_beh):
        super().__init__()
        self.bert=DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.text_proj=nn.Sequential(nn.Linear(768,256), nn.ReLU(), nn.Dropout(0.3))
        self.beh=nn.Sequential(nn.Linear(n_beh,64), nn.ReLU(), nn.Dropout(0.3),
                               nn.Linear(64,32), nn.ReLU())
        self.head=nn.Sequential(nn.Linear(256+32,64), nn.ReLU(), nn.Dropout(0.4), nn.Linear(64,1))
    def forward(self, ids, mask, beh):
        cls=self.bert(input_ids=ids, attention_mask=mask).last_hidden_state[:,0]   # [CLS]
        t=self.text_proj(cls); b=self.beh(beh)
        return self.head(torch.cat([t,b], dim=1)).squeeze(-1)

# Layer-wise LR decay (LLRD): deeper DistilBERT layers + new heads train faster,
# embeddings slowest — the project's standard fine-tuning recipe.
def make_optimizer(m, base_lr=LR, head_lr=1e-3, decay=0.95, wd=0.01):
    b=m.bert; n=len(b.transformer.layer); g=[]
    g.append({'params':list(b.embeddings.parameters()), 'lr':base_lr*(decay**n)})
    for i,layer in enumerate(b.transformer.layer):
        g.append({'params':list(layer.parameters()), 'lr':base_lr*(decay**(n-1-i))})
    g.append({'params':list(m.text_proj.parameters())+list(m.beh.parameters())+list(m.head.parameters()),
              'lr':head_lr})
    return torch.optim.AdamW(g, weight_decay=wd)

# ── FOCAL LOSS (replaces weighted BCE) ──────────────────────────────────────
# Binary focal loss (Lin et al. 2017) on the single sigmoid logit. gamma down-weights easy
# examples; alpha weights the FAKE class (alpha=fake-class weight). BOTH are SWEPT in cell-5 —
# the focal paper found a LOW alpha for the rare class can beat a high one (gamma already
# handles the easy/hard imbalance), so we tune rather than hard-set alpha=genuine-fraction.
def focal_loss(logit, target, gamma, alpha):
    p = torch.sigmoid(logit)
    p_t = p * target + (1 - p) * (1 - target)
    a_t = alpha * target + (1 - alpha) * (1 - target)      # alpha applied to fake(=1)
    bce = F.binary_cross_entropy_with_logits(logit, target, reduction='none')
    return (a_t * (1 - p_t) ** gamma * bce).mean()

# ── reusable train/eval helpers (a fresh model per call, so the sweep + final share code) ──
USE_AMP = (device == 'cuda')          # mixed precision on GPU (faster), plain on CPU

def predict(m, dl):
    m.eval(); probs=[]
    with torch.no_grad():
        for ids,mask,beh,y,w in dl:
            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    logit=m(ids.to(device), mask.to(device), beh.to(device))
            else:
                logit=m(ids.to(device), mask.to(device), beh.to(device))
            probs.append(torch.sigmoid(logit.float()).cpu().numpy())
    return np.concatenate(probs)

def best_thr(y, p):
    bf,bt=-1,0.5
    for t in np.linspace(0.05,0.95,19):
        s=f1_score(y,(p>=t).astype(int),average='macro')
        if s>bf: bf,bt=s,t
    return bt,bf

def fit_once(gamma, alpha, train_dl, epochs, patience=3, tag=''):
    """Train a fresh model with focal(gamma,alpha); early-stop on val macro-F1; restore best.
    Returns (best_model, best_thr, best_val_f1, train_hist, val_hist)."""
    m=MultiModalDistilBert(len(feat_cols)).to(device)
    opt=make_optimizer(m)
    total=len(train_dl)*epochs
    sched=get_linear_schedule_with_warmup(opt, int(0.1*total), total)   # 10% warmup
    sc_amp=torch.amp.GradScaler('cuda', enabled=USE_AMP)
    best_f1, best_state, best_t, wait = -1, None, 0.5, 0
    trh, vah = [], []
    for ep in range(epochs):
        m.train(); tot=0
        pbar=tqdm(train_dl, desc=f'{tag} ep{ep+1}/{epochs}', leave=False)
        for ids,mask,beh,y,w in pbar:
            opt.zero_grad()
            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    logit=m(ids.to(device), mask.to(device), beh.to(device))
                    loss=focal_loss(logit, y.to(device), gamma, alpha)
                sc_amp.scale(loss).backward(); sc_amp.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                sc_amp.step(opt); sc_amp.update()
            else:
                logit=m(ids.to(device), mask.to(device), beh.to(device))
                loss=focal_loss(logit, y.to(device), gamma, alpha)
                loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
            sched.step(); tot+=loss.item()
            pbar.set_postfix(loss=f'{loss.item():.3f}')
        p_va=predict(m, va_dl); thr,f1=best_thr(va.label.values, p_va)
        trh.append(tot/len(train_dl)); vah.append(f1)
        print(f'  {tag} ep{ep+1}: loss={tot/len(train_dl):.4f}  val_macroF1={f1:.4f} @thr {thr:.2f}')
        if f1>best_f1:
            best_f1, best_t = f1, thr
            best_state={k:v.cpu().clone() for k,v in m.state_dict().items()}; wait=0
        else:
            wait+=1
            if wait>=patience: print('   early stop'); break
    m.load_state_dict(best_state)
    return m, best_t, best_f1, trh, vah

print('params:', sum(p.numel() for p in MultiModalDistilBert(len(feat_cols)).parameters())/1e6, 'M')

In [ ]:
import itertools
# ============================ PHASE A — hyperparameter sweep ============================
# Tune focal (gamma, alpha) on a FAST PROXY: a stratified train subsample + few epochs, scored
# on the FULL balanced val set (fair selection). Cheap relative to a full 642k run per grid point.
PROXY_N      = 120_000
PROXY_EPOCHS = 2
GAMMAS = [1.0, 2.0, 3.0]            # focusing strength (down-weight easy examples)
ALPHAS = [0.25, 0.5, 0.75]         # fake-class weight; low can beat high (Lin et al.)

rng = np.random.default_rng(42)
idx = rng.choice(len(tr), size=min(PROXY_N, len(tr)), replace=False)   # preserves ~11% fake rate
proxy_dl = DataLoader(MMData(tr.text.values[idx], Btr[idx], tr.label.values[idx]),
                      batch_size=BATCH, shuffle=True)
print(f'PROXY sweep: {len(idx)} rows (fake rate {tr.label.values[idx].mean():.3f}) | '
      f'{len(GAMMAS)*len(ALPHAS)} configs x {PROXY_EPOCHS} epochs\n')

sweep=[]
for g,a in itertools.product(GAMMAS, ALPHAS):
    _, _, f1, _, _ = fit_once(g, a, proxy_dl, PROXY_EPOCHS, patience=2, tag=f'[g{g} a{a}]')
    sweep.append((f1, g, a))
    print(f'  config gamma={g} alpha={a}  ->  proxy val macroF1={f1:.4f}\n')
sweep.sort(reverse=True)
print('PROXY ranking (best first):')
for f1,g,a in sweep: print(f'   gamma={g} alpha={a}: {f1:.4f}')
BEST_F1, BEST_GAMMA, BEST_ALPHA = sweep[0]
print(f'\n==> BEST proxy config: gamma={BEST_GAMMA}, alpha={BEST_ALPHA}  (proxy val {BEST_F1:.4f})')

# ============================ PHASE B — final full-data train ============================
# Retrain from scratch at the best config on the FULL train data, full epochs. Sets the globals
# used by the eval/export cells below: model, best_thr_v, tr_hist, val_hist.
print('\n=============== FINAL: full-data train @ best config ===============')
model, best_thr_v, final_f1, tr_hist, val_hist = fit_once(
    BEST_GAMMA, BEST_ALPHA, tr_dl, EPOCHS, patience=3, tag='FINAL')
print(f'FINAL full-data val macroF1={final_f1:.4f} @thr {best_thr_v:.2f} '
      f'(gamma={BEST_GAMMA}, alpha={BEST_ALPHA})')
FOCAL_GAMMA, FOCAL_ALPHA = BEST_GAMMA, BEST_ALPHA   # recorded into meta on export

In [ ]:
# Training curves
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(range(1,len(tr_hist)+1),tr_hist,marker='o'); ax[0].set_title('Train loss'); ax[0].set_xlabel('epoch')
ax[1].plot(range(1,len(val_hist)+1),val_hist,marker='o',color='green'); ax[1].set_title('Val macro-F1'); ax[1].set_xlabel('epoch')
ax[1].axvline(int(np.argmax(val_hist))+1, ls='--', color='red', label='best epoch'); ax[1].legend()
plt.tight_layout(); plt.savefig('yelp_multimodal_distilbert_training.png', dpi=120); plt.show()

In [ ]:
p_te=predict(model, te_dl); pred=(p_te>=best_thr_v).astype(int); y_te=te.label.values
print(f'best focal config: gamma={FOCAL_GAMMA}, alpha={FOCAL_ALPHA}')
print('threshold =', round(best_thr_v,2))
print('test accuracy  =', round(accuracy_score(y_te,pred),4))
print('test macro-F1  =', round(f1_score(y_te,pred,average='macro'),4))
print('fake recall    =', round(recall_score(y_te,pred,pos_label=1),3))
print('fake precision =', round(precision_score(y_te,pred,pos_label=1),3))
print('genuine spec   =', round(recall_score(y_te,pred,pos_label=0),3))
print('\nReferences (same Yelp test): text-only ~69% · pure sklearn Model B (full) ens 76.2% / xgb 77.3% · standard Model B DL ~76.7%')
print()
print(classification_report(y_te, pred, target_names=['genuine','fake'], digits=4))

In [ ]:
# Export to ONNX (thread-safe serving) + bundle scaler/threshold/tokenizer
!pip install -q onnx onnxruntime onnxscript
import os, json, joblib, torch, zipfile, numpy as np
model.eval()
dummy_ids =torch.zeros(1,MAX_LEN,dtype=torch.long,device=device)
dummy_mask=torch.ones(1,MAX_LEN,dtype=torch.long,device=device)
dummy_beh =torch.zeros(1,len(feat_cols),dtype=torch.float32,device=device)
_args=(dummy_ids,dummy_mask,dummy_beh)
_kw=dict(input_names=['input_ids','attention_mask','beh'], output_names=['logit'],
         dynamic_axes={'input_ids':{0:'b'},'attention_mask':{0:'b'},'beh':{0:'b'},'logit':{0:'b'}},
         opset_version=14, do_constant_folding=True)
try:
    torch.onnx.export(model,_args,'yelp_multimodal_distilbert.onnx',dynamo=False,**_kw)  # stable legacy exporter
except TypeError:
    torch.onnx.export(model,_args,'yelp_multimodal_distilbert.onnx',**_kw)               # older torch: legacy is default
joblib.dump(sc,'yelp_multimodal_distilbert_scaler.joblib')
tok.save_pretrained('yelp_multimodal_distilbert_tok')   # HF tokenizer = a folder of files
json.dump({'threshold':float(best_thr_v),'feat_cols':feat_cols,'max_len':MAX_LEN,
           'focal_gamma':float(FOCAL_GAMMA),'focal_alpha':float(FOCAL_ALPHA)},
          open('yelp_multimodal_distilbert_meta.json','w'), indent=2)

# sanity check: ONNX vs PyTorch on 4 test rows
import onnxruntime as ort
sess=ort.InferenceSession('yelp_multimodal_distilbert.onnx',providers=['CPUExecutionProvider'])
enc=tok(list(te.text.values[:4]),truncation=True,padding='max_length',max_length=MAX_LEN,return_tensors='np')
onnx_logit=sess.run(None,{'input_ids':enc['input_ids'].astype('int64'),
                          'attention_mask':enc['attention_mask'].astype('int64'),
                          'beh':Bte[:4]})[0].flatten()
print('ONNX inputs:', [i.name for i in sess.get_inputs()])
print('ONNX prob :', np.round(1/(1+np.exp(-onnx_logit)),4))
print('Torch prob:', np.round(p_te[:4],4))

with zipfile.ZipFile('yelp_multimodal_distilbert_focal_onnx.zip','w',zipfile.ZIP_DEFLATED) as zf:
    for fn in ['yelp_multimodal_distilbert.onnx','yelp_multimodal_distilbert_scaler.joblib',
               'yelp_multimodal_distilbert_meta.json']:
        zf.write(fn)
    for f in os.listdir('yelp_multimodal_distilbert_tok'):   # bundle the tokenizer folder INSIDE the one zip
        zf.write(os.path.join('yelp_multimodal_distilbert_tok', f), arcname=f'tokenizer/{f}')
from google.colab import files
files.download('yelp_multimodal_distilbert_focal_onnx.zip')  # extract -> data/yelp_multimodal_distilbert_focal_onnx/
print('done: one zip -> onnx + scaler + meta + tokenizer/  (extract -> data/yelp_multimodal_distilbert_focal_onnx/)')